# Week 13, Lab 2 — Regression and model evaluation

**HWRS 564a · Thursday, November 19, 2026 · 75-minute lecture with embedded lab**

**Question:** Where do the recovery slopes come from, and what do their errors reveal?

This notebook runs independently of Tuesday's notebook. You will reproduce one site's slope difference, inspect a statistical trend's residuals, evaluate groundwater-model errors, and assemble an evidence figure. No network, MODFLOW run, or large model archive is required.

There are four short coding exercises. Interpret the plots in markdown, then restart and run all before saving.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works from the repository root, this lab folder, or its generated solution folder.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/cache/pinal_model_head_obs.csv").exists())
BLUE, RED = "#0C234B", "#AB0520"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False})

In [ ]:
heads = pd.read_csv(ROOT / "data/cache/pinal_model_head_obs.csv", dtype={"site": str})
paired = pd.read_csv(ROOT / "data/week13_pinal_recovery_slopes.csv", dtype={"site": str})
period = heads.loc[heads["year"].ge(1985) & heads["year"].lt(2005)].copy()
SITE = "A3273OC"
one = period.loc[period["site"].eq(SITE)].sort_values("year").copy()
assert len(one) == 36 and np.isfinite(one[["year", "observed", "simulated"]]).all().all()
print(f"{SITE}: {len(one)} observations, {one.year.min():.2f}–{one.year.max():.2f}")
one.head()

## 1 — A line is a statistical model · minutes 0–20

Tuesday estimated a mean difference of about +1.44 ft/year across 191 qualifying sites. That is a difference in recovery slopes, not a difference in absolute heads.

For one well, a straight-line approximation is

$$h_i=a+b(t_i-1985)+\epsilon_i.$$

- $a$ is fitted head in 1985, in feet. If 1985 is outside this site's record, it is an extrapolated intercept.
- $b$ is head change per year, in ft/year; positive means rising water.
- $\epsilon_i$ represents departures from the straight-line mean model.

Least squares chooses $a$ and $b$ to minimize the sum of squared vertical residuals. It does not require the groundwater history to be physically linear. Whether a line is adequate is a separate question.

### YOUR TURN 1 — Reproduce one paired slope · minutes 20–35

Set `x` to years since 1985. Fit observed and simulated heads with `np.polyfit(x, values, 1)`, which returns slope then intercept. Store their slopes and intercepts in `b_obs, a_obs` and `b_sim, a_sim`; calculate `site_gap` as observed slope minus simulated slope.

In [ ]:
# YOUR TURN
x = ...
b_obs, a_obs = ...
b_sim, a_sim = ...
site_gap = ...

In [ ]:
# CHECK
reference = paired.set_index("site").loc[SITE]
assert np.allclose(x, one["year"].to_numpy() - 1985)
assert np.isclose(b_obs, reference["observed_slope_ftyr"])
assert np.isclose(b_sim, reference["simulated_slope_ftyr"])
assert np.isclose(site_gap, reference["gap_ftyr"])
assert np.isclose(a_obs, one["observed"].mean() - b_obs * x.mean())
assert np.isclose(a_sim, one["simulated"].mean() - b_sim * x.mean())
print(f"Observed {b_obs:+.2f}, simulated {b_sim:+.2f}, gap {site_gap:+.2f} ft/year")

In [ ]:
fitted_obs = a_obs + b_obs * x
fitted_sim = a_sim + b_sim * x
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(one["year"], one["observed"], color=BLUE, label="observed head", s=22)
ax.plot(one["year"], one["simulated"], color=RED, marker="s", ms=3, label="simulated head")
ax.plot(one["year"], fitted_obs, color=BLUE, ls="--", label="observed linear trend")
ax.plot(one["year"], fitted_sim, color=RED, ls=":", label="simulated linear trend")
ax.set(xlabel="year", ylabel="head elevation (ft)", title=f"{SITE}: a line summarizes an unevenly sampled history")
ax.legend(frameon=False, ncol=2, fontsize=9)
plt.show()

**Interpret:** Does this site's slope difference have the same sign as the mean across all sites? Does the head record make a single linear rate convincing? Notice the long gap before dense quarterly sampling.

For a linear model with an intercept, you can also recover the slope from centered sums:

$$b=\frac{\sum (x_i-\bar x)(h_i-\bar h)}{\sum (x_i-\bar x)^2},\qquad a=\bar h-b\bar x.$$

The numerator measures how time and head vary together. The denominator measures the spread of observation times. An isolated early observation can have substantial leverage.

## 2 — Residual diagnostics · minutes 35–50

A **trend residual** is `observed head - fitted statistical trend`. It asks where the line fails to describe the observed record.

### YOUR TURN 2 — Calculate residuals and their sequence dependence

Calculate `trend_residual`. Then calculate `adjacent_r` as the correlation between successive residuals in time order using `np.corrcoef(residual[:-1], residual[1:])[0, 1]`.

The observations are irregularly spaced: this is correlation between adjacent records, **not** a fixed one-year or one-quarter autocorrelation.

In [ ]:
# YOUR TURN
trend_residual = ...
adjacent_r = ...

In [ ]:
# CHECK
expected_residual = one["observed"].to_numpy() - fitted_obs
assert np.allclose(trend_residual, expected_residual)
assert abs(np.mean(trend_residual)) < 1e-8
assert np.isclose(adjacent_r, np.corrcoef(expected_residual[:-1], expected_residual[1:])[0, 1])
print(f"Adjacent-record residual correlation: {adjacent_r:+.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].scatter(one["year"], trend_residual, color=BLUE, s=20)
axes[0].set(xlabel="year", ylabel="trend residual (ft)", title="Pattern through time")
axes[1].scatter(fitted_obs, trend_residual, color=BLUE, s=20)
axes[1].set(xlabel="fitted head (ft)", title="Pattern versus fitted head")
axes[2].scatter(trend_residual[:-1], trend_residual[1:], color=BLUE, s=20)
axes[2].set(xlabel="residual at record i (ft)", ylabel="residual at record i+1 (ft)", title="Adjacent records")
for ax in axes[:2]: ax.axhline(0, color="0.5", lw=1)
fig.tight_layout()
plt.show()

**Interpret:** Look for curvature, changing spread, or runs of the same sign. A small adjacent correlation would not prove independence or establish that the line is appropriate. Does the record suggest a constant recovery rate?

An ordinary regression interval for a slope needs assumptions about the errors, including independence for the usual formula. Correlation can make ordinary uncertainty calculations too optimistic. A bootstrap of individual time-series rows also breaks their dependence.

A **confidence interval for a mean response** describes uncertainty in the fitted mean. A **prediction interval for a new observation** must also include scatter around that mean and is generally wider. Neither automatically includes all uncertainty in a physical groundwater model.

## 3 — Groundwater-model error · minutes 50–65

Now compare MODFLOW's simulated heads directly with the measured heads. Define

$$e_i=h_{sim,i}-h_{obs,i}.$$

Positive error means the simulated head is too high. This differs from Tuesday's slope-gap sign and from a trend residual. Keep the labels explicit.

### YOUR TURN 3 — Evaluate one site's head errors

Calculate `model_error`, `bias`, `mae`, and `rmse` for this site's 36 paired observations. Use the definitions:

$$bias=\overline e,\quad MAE=\overline{|e|},\quad RMSE=\sqrt{\overline{e^2}}.$$

All three summaries have units feet. Bias can hide cancellation; RMSE gives large errors more influence.

In [ ]:
# YOUR TURN
model_error = ...
bias = ...
mae = ...
rmse = ...

In [ ]:
# CHECK
expected_error = one["simulated"].to_numpy() - one["observed"].to_numpy()
assert np.allclose(model_error, expected_error)
assert np.isclose(bias, expected_error.mean())
assert np.isclose(mae, np.abs(expected_error).mean())
assert np.isclose(rmse, np.sqrt(np.mean(expected_error**2)))
assert rmse >= mae >= abs(bias) - 1e-10
print(f"{SITE}: bias {bias:+.1f} ft, MAE {mae:.1f} ft, RMSE {rmse:.1f} ft")

### YOUR TURN 4 — High correlation with poor agreement

In a synthetic example, let `synthetic_obs` run from 100 to 200 ft in 20 equally spaced steps, and let `synthetic_sim` be exactly 50 ft higher at every point. Calculate `synthetic_r` and `synthetic_rmse`.

**Predict before running:** will correlation expose the 50 ft error?

In [ ]:
synthetic_obs = np.linspace(100, 200, 20)
# YOUR TURN
synthetic_sim = ...
synthetic_r = ...
synthetic_rmse = ...

In [ ]:
# CHECK
assert np.allclose(synthetic_sim, synthetic_obs + 50)
assert np.isclose(synthetic_r, 1)
assert np.isclose(synthetic_rmse, 50)
print(f"Synthetic correlation = {synthetic_r:.2f}; RMSE = {synthetic_rmse:.0f} ft")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, obs, sim, title in [
    (axes[0], one["observed"], one["simulated"], SITE),
    (axes[1], synthetic_obs, synthetic_sim, "Synthetic: r = 1, offset = 50 ft")]:
    lo, hi = min(np.min(obs), np.min(sim)), max(np.max(obs), np.max(sim))
    ax.plot([lo, hi], [lo, hi], color="0.5", ls="--", label="1:1 agreement")
    ax.scatter(obs, sim, color=BLUE, s=22)
    ax.set(xlabel="observed head (ft)", ylabel="simulated head (ft)", title=title)
    ax.set_aspect("equal", adjustable="box")
    ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

## 4 — An evidence figure and a bounded claim · minutes 65–75

The supplied figure combines the individual site's evidence with the distribution across sites. Change labels or annotations to clarify your conclusion. Avoid claiming that this selected well represents the entire basin.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].scatter(one["year"], one["observed"], s=20, color=BLUE, label="observed")
axes[0].plot(one["year"], one["simulated"], color=RED, ls="--", marker="s", ms=3, label="simulated")
axes[0].set(xlabel="year", ylabel="head elevation (ft)", title=f"a. {SITE}")
axes[0].legend(frameon=False, fontsize=9)
axes[1].scatter(one["year"], model_error, color=BLUE, s=20)
axes[1].axhline(0, color="0.5", lw=1)
axes[1].set(xlabel="year", ylabel="simulated − observed head (ft)", title="b. Head error through time")
axes[2].hist(paired["gap_ftyr"], bins=25, color=BLUE)
axes[2].axvline(0, color="0.5", lw=1)
axes[2].axvline(paired["gap_ftyr"].mean(), color=RED, ls="--", label="mean gap +1.44")
axes[2].set(xlabel="observed − simulated slope (ft/year)", ylabel="sites", title="c. All 191 qualifying sites")
axes[2].legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

# Optional export: use a reproducible output directory; do not submit data/model files.
OUT = ROOT / "_run/week13_figures"
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "pinal_evidence.pdf", bbox_inches="tight")
fig.savefig(OUT / "pinal_evidence.png", dpi=300, bbox_inches="tight")

Write no more than 200 words answering:

1. What does the distribution of paired recovery slopes show, and how does this site differ from the average?
2. What do the trend residuals and head errors reveal that an average slope or RMSE conceals?
3. Why does agreement with calibration observations provide limited evidence for predictions to 2115?
4. What additional evidence would strengthen a claim about future responses to reduced pumping?

**Your assessment:**

*Write here.*

These HOB observations supported model calibration. The exercise evaluates historical calibration performance; splitting these archived outputs after calibration does not create independent validation. A genuine predictive evaluation requires observations withheld from model development or new data, matched to the intended prediction task. Forecast evaluation should respect time order; a random row split can mix closely related observations.

A rate discrepancy cannot identify whether pumping, recharge, storage, boundaries, or another feature caused it. Nor does it establish the direction of every future prediction error.

**Before leaving:** restart the kernel, run all, inspect the figure labels, and save. Both labs are in-class practice; submission instructions and deadlines remain in D2L.

## Sources

Pinal HOB cache and qualification rule: the Week 12 lab and `scripts/prepare_week13_data.py` in this repository.

Conceptual references: *Data Analysis in Water Sciences*, Steven Pestana, Jessica D. Lundquist, and contributors: [linear regression](https://mountain-hydrology-research-group.github.io/data-analysis/modules/module4/lab4-1.html), [confidence intervals](https://mountain-hydrology-research-group.github.io/data-analysis/modules/module4/lab4-3.html), and [autocorrelation](https://mountain-hydrology-research-group.github.io/data-analysis/modules/module5/lab5-2.html).